# PySpark & MLlib

## Fetching the Data

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s titanic
!ls -lh titanic/

## Importing the Required Libraries

In [ ]:
!pip install inflection pyspark

In [ ]:
from pathlib import Path

import inflection
import pyspark
import pyspark.ml.feature as MF
import pyspark.sql.functions as F

## Loading the Data

The csv file is loaded into a spark `Dataframe`.

In [ ]:
data_dir = Path("titanic")
csv_path = data_dir / "train.csv"

In [ ]:
spark = (pyspark.sql.SparkSession.builder.appName("PySpark demonstration")
                                         .getOrCreate())
df = spark.read.csv(str(csv_path), header=True, inferSchema=True)

In [ ]:
passengers_count = df.count()

In [ ]:
print(passengers_count)

In [ ]:
df.show(5)

In [ ]:
df.describe().show()

In [ ]:
df.printSchema()

## Renaming the Columns to Follow Python's Snake Case Convention

In [ ]:
df = df.toDF(*(inflection.underscore(c) for c in df.columns))
df.show(5)

## Exploring the Data

In [ ]:
df.select("survived", "pclass", "embarked").show()

In [ ]:
df.groupBy("survived").count().show()

In [ ]:
df.groupBy("sex", "survived").count().show()

In [ ]:
df.groupBy("pclass", "survived").count().show()

## Preparing the Data

### Null Values

In [ ]:
def null_value_count(df: pyspark.sql.DataFrame):
  null_columns_counts = []
  for k in df.columns:
    null_rows = df.where(F.col(k).isNull()).count()
    if null_rows:
      null_columns_counts.append((k, null_rows))
  return null_columns_counts

null_columns_count_list = null_value_count(df)
spark.createDataFrame(null_columns_count_list,
                      ['column_with_nulls', 'nulls_count']).show()

### Analyzing the Passengers' Age

In [ ]:
mean_age = df.select(F.mean('age')).collect()[0][0]
print(mean_age)

We could use this average age to impute the unknown ages, but we'll see how to go a bit further and compute the average age per title.

In [ ]:
df = df.withColumn("title", F.regexp_extract(F.col("name"), r"([A-Za-z]+)\.", 1))
df.show()

In [ ]:
df.select("title").distinct().show()

In [ ]:
replacements = dict(Miss=["Mlle", "Ms"],
                    Mr=["Dr", "Major", "Capt", "Sir", "Don"],
                    Mrs=["Lady", "Countess", "Mme"],
                    Other=["Jonkheer", "Col", "Rev"])
df = df.replace({name: title
                 for title, names in replacements.items()
                 for name in names},
                subset=["title"])

In [ ]:
df.select("title").distinct().show()

In [ ]:
average_title_ages = df.groupby("title").agg(F.avg("age").alias("average_age"))
average_title_ages.show()

### Imputing the Age

Each passenger gets the average age of the group of passengers sharing the same title.

In [ ]:
df = (df.join(average_title_ages, "title")
        .withColumn("age", F.when(F.col("age").isNull(),
                                  F.col("average_age"))
                            .otherwise(F.col("age")))
        .drop("average_age"))
df.show()

The *embarked* feature (the port of embarkation) has only two missing values. We'll impute the mode for these two values.

In [ ]:
df.groupBy("embarked").count().show()

In [ ]:
df = df.na.fill(dict(embarked="S"))

The *cabin* feature can be dropped: the cabin number is almost unique for each passenger in a cabin, and would quickly lead to overfitting.

In [ ]:
df = df.drop("cabin")

In [ ]:
df.printSchema()

### Adding a New Feature.

The family size.

In [ ]:
df = df.withColumn("family_size", F.col("sib_sp") + F.col("parch") + 1)

In [ ]:
df.groupBy("family_size").count().show()

In [ ]:
df = df.withColumn("alone", (F.col("family_size") == 1).cast("long"))
df.show()

In [ ]:
df.columns

### Encoding Categorical Variables

In [ ]:
indexers = [MF.StringIndexer(inputCol=column,
                             outputCol=f"{column}_index").fit(df)
            for column in ["sex", "embarked", "title"]]
pipeline = pyspark.ml.Pipeline(stages=indexers)
df = pipeline.fit(df).transform(df)
df.show()

In [ ]:
df.printSchema()

### Dropping the Non-Numerical Features

In [ ]:
df = df.drop(*(name for name, dtype in df.dtypes if dtype == "string"))

In [ ]:
df.show()

### Creating the Feature Vectors

In [ ]:
feature = MF.VectorAssembler(
    inputCols=df.drop("survived", "passenger_id").columns, outputCol="features")
feature_vector = feature.transform(df)

In [ ]:
feature_vector.show()

### Splitting Into Training and Test Sets

In [ ]:
training_data, test_data = feature_vector.randomSplit([0.8, 0.2], seed=11)

## Training Models

### Logistic Regression

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(labelCol="survived", featuresCol="features")
lr_model = lr.fit(training_data)
lr_prediction = lr_model.transform(test_data)
lr_prediction.select("prediction", "survived", "features").show()
evaluator = pyspark.ml.evaluation.MulticlassClassificationEvaluator(
    labelCol="survived", predictionCol="prediction", metricName="accuracy")

# Evaluation
lr_accuracy = evaluator.evaluate(lr_prediction)
print(f"Logistic regression accuracy: {lr_accuracy:.2f}")

### Random Forest

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rfc = RandomForestClassifier(labelCol="survived", featuresCol="features")
rfc_model = rfc.fit(training_data)
rfc_predictions = rfc_model.transform(test_data)
rfc_predictions.select("prediction", "survived", "features").show()

# Evaluation
rfc_accuracy = evaluator.evaluate(rfc_predictions)
print(f"Random forest accuracy: {rfc_accuracy:.2f}")

### Boosted Tree

In [ ]:
from pyspark.ml.classification import GBTClassifier

gbtc = GBTClassifier(labelCol="survived", featuresCol="features", maxIter=10)
gbtc_model = gbtc.fit(training_data)
gbtc_predictions = gbtc_model.transform(test_data)
gbtc_predictions.select("prediction", "survived", "features").show()

# Evaluation
gbtc_accuracy = evaluator.evaluate(gbtc_predictions)
print(f"Boosted tree accuracy: {gbtc_accuracy:.2f}")

### Support Vector Machine

In [ ]:
from pyspark.ml.classification import LinearSVC

svmc = LinearSVC(labelCol="survived", featuresCol="features")
svmc_model = svmc.fit(training_data)
svmc_predictions = svmc_model.transform(test_data)
svmc_predictions.select("prediction", "survived", "features").show()

# Evaluation
svmc_accuracy = evaluator.evaluate(svmc_predictions)
print(f"Support vector machine accuracy: {svmc_accuracy:.2f}")

## Reference

https://spark.apache.org/docs/latest/ml-classification-regression.html